In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

In [2]:
df=pd.read_excel('2024 Democracy Index.xlsx')
df.head()

,Entity,Code,World region according to OWID,Year,Democracy Index,Regime Classification Number,Regime Classification,Civil Liberties Index,Democratic Culture Index,Electoral Pluralism Index,Functioning Government Index,Political Participation Index
0,Afghanistan,AFG,Asia,2006,3.06,0,Authoritarian regime,4.41,2.5,6.17,0.00,2.22
1,Afghanistan,AFG,Asia,2008,3.02,0,Authoritarian regime,4.41,2.5,5.17,0.79,2.22
2,Afghanistan,AFG,Asia,2010,2.48,0,Authoritarian regime,3.82,2.5,2.50,0.79,2.78
3,Afghanistan,AFG,Asia,2011,2.48,0,Authoritarian regime,3.82,2.5,2.50,0.79,2.78
4,Afghanistan,AFG,Asia,2012,2.48,0,Authoritarian regime,3.82,2.5,2.50,0.79,2.78


In [3]:
df.shape

(2839, 12)

In [4]:
df.describe()

,Year,Democracy Index,Regime Classification Number,Civil Liberties Index,Democratic Culture Index,Electoral Pluralism Index,Functioning Government Index,Political Participation Index
count,2839.000000,2839.000000,2839.00000,2839.000000,2839.000000,2839.000000,2839.000000,2839.000000
mean,2015.823529,5.437196,1.26770,5.902455,5.536636,5.843815,4.850391,5.056365
std,5.194070,2.249399,1.06295,2.774822,1.727107,3.639116,2.539509,1.960076
min,2006.000000,0.250000,0.00000,0.000000,1.250000,-0.330000,0.000000,0.000000
25%,2012.000000,3.405000,0.00000,3.530000,4.380000,2.500000,2.860000,3.890000
50%,2016.000000,5.730000,1.00000,6.180000,5.630000,7.000000,5.000000,5.000000
75%,2020.000000,7.240000,2.00000,8.530000,6.250000,9.170000,6.790000,6.670000
max,2024.000000,9.930000,3.00000,10.000000,10.000000,10.000000,10.000000,10.000000


In [5]:
df.dtypes

Entity                             object
Code                               object
World region according to OWID     object
Year                                int64
Democracy Index                   float64
Regime Classification Number        int64
Regime Classification              object
Civil Liberties Index             float64
Democratic Culture Index          float64
Electoral Pluralism Index         float64
Functioning Government Index      float64
Political Participation Index     float64
dtype: object

In [6]:
df['Regime Classification'].unique()

array(['Authoritarian regime', 'Hybrid regime', 'Flawed democracy',
       'Full democracy'], dtype=object)

In [7]:
df['Entity'].nunique()

167

So we have data of 167 countries in total from year 2006 to 2024.
Now let's explore what these indices actually mean, used claude for this part as column descriptions were not available in the original dataset-

### Electoral Pluralism Index — 
Whether elections are free and fair. Covers voter registration, multi-party competition, electoral integrity, and whether opposition parties can actually operate and win. It's the most foundational pillar — you can't be a democracy without it.

### Civil Liberties Index — 
The freedoms people have in daily life: press freedom, freedom of speech and assembly, protection from arbitrary detention, an independent judiciary, and the rule of law. A country can hold elections but still score poorly here if dissent is suppressed.

### Functioning Government Index — 
Whether the government actually governs effectively and democratically. Looks at how much elected officials control policy (vs. the military, oligarchs, or foreign powers), government transparency, corruption levels, and citizens' confidence in institutions.

### Political Participation Index — 
How engaged citizens are beyond just voting. Includes membership in political parties and civil society organizations, interest in politics, and whether women and minorities have meaningful political representation. Low scores here often signal apathy or deliberate exclusion.

### Democratic Culture Index — 
The most subjective of the five. Measures whether society believes in democracy — do people support it over strong-man rule? Is there tolerance for minority views? Do citizens separate civil from religious authority? This captures whether democratic institutions have genuine public backing or are just a thin facade.

In [11]:
numeric_df = df.select_dtypes(include=[np.number])

# Remove missing values
numeric_df = numeric_df.dropna()

# ============================================
# PEARSON CORRELATION MATRIX
# ============================================

corr_matrix = numeric_df.corr(method='pearson')

print("\n========== PEARSON CORRELATION MATRIX ==========\n")
print(corr_matrix)

# ============================================
# P-VALUE MATRIX
# ============================================

cols = numeric_df.columns

pval_matrix = pd.DataFrame(
    np.ones((len(cols), len(cols))),
    columns=cols,
    index=cols
)

for i in range(len(cols)):
    for j in range(len(cols)):

        if i == j:
            pval_matrix.iloc[i, j] = 0

        else:
            corr, pval = pearsonr(
                numeric_df.iloc[:, i],
                numeric_df.iloc[:, j]
            )

            pval_matrix.iloc[i, j] = pval

print("\n========== P-VALUE MATRIX ==========\n")
print(pval_matrix)

# ============================================
# DEMOCRACY INDEX RELATIONSHIPS
# ============================================

TARGET_COL = "Democracy Index"   # change if needed

print(f"\n========== VARIABLES RELATED TO {TARGET_COL} ==========\n")

results = []

for col in cols:

    if col == TARGET_COL:
        continue

    corr, pval = pearsonr(
        numeric_df[TARGET_COL],
        numeric_df[col]
    )

    results.append({
        "Variable": col,
        "Correlation": corr,
        "P_Value": pval,
        "Significant": pval < 0.05
    })

results_df = pd.DataFrame(results)

# Sort by absolute correlation strength
results_df["Abs_Corr"] = results_df["Correlation"].abs()

results_df = results_df.sort_values(
    by="Abs_Corr",
    ascending=False
)

print(results_df[
    ["Variable", "Correlation", "P_Value", "Significant"]
])

# ============================================
# OPTIONAL: SPEARMAN TESTS
# ============================================

print("\n========== SPEARMAN CORRELATIONS ==========\n")

spearman_results = []

for col in cols:

    if col == TARGET_COL:
        continue

    corr, pval = spearmanr(
        numeric_df[TARGET_COL],
        numeric_df[col]
    )

    spearman_results.append({
        "Variable": col,
        "Spearman_Corr": corr,
        "P_Value": pval,
        "Significant": pval < 0.05
    })

spearman_df = pd.DataFrame(spearman_results)

spearman_df["Abs_Corr"] = spearman_df["Spearman_Corr"].abs()

spearman_df = spearman_df.sort_values(
    by="Abs_Corr",
    ascending=False
)

print(spearman_df[
    ["Variable", "Spearman_Corr", "P_Value", "Significant"]
])


========== PEARSON CORRELATION MATRIX ==========

                                   Year  Democracy Index  \
Year                           1.000000        -0.042551   
Democracy Index               -0.042551         1.000000   
Regime Classification Number  -0.029669         0.954443   
Civil Liberties Index         -0.126118         0.948185   
Democratic Culture Index      -0.069636         0.719921   
Electoral Pluralism Index     -0.047408         0.935588   
Functioning Government Index  -0.051834         0.916293   
Political Participation Index  0.151093         0.837132   

                               Regime Classification Number  \
Year                                              -0.029669   
Democracy Index                                    0.954443   
Regime Classification Number                       1.000000   
Civil Liberties Index                              0.905552   
Democratic Culture Index                           0.686661   
Electoral Pluralism Index     